<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/dev/MeteoData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import geopandas as gpd
import pandas as pd
import seaborn as sb
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point
from shapely.geometry.polygon import Polygon
from matplotlib.colors import ListedColormap

In [ ]:
estaciones_df= pd.read_csv('/content/drive/MyDrive/eco2026/Estaciones_IDEAM_20260527.csv')

In [ ]:
estaciones_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9464 entries, 0 to 9463
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Codigo                9464 non-null   int64  
 1   Nombre                9464 non-null   object 
 2   Categoria             9464 non-null   object 
 3   Tecnologia            9464 non-null   object 
 4   Estado                9464 non-null   object 
 5   Departamento          9464 non-null   object 
 6   Municipio             9464 non-null   object 
 7   Ubicación             9464 non-null   object 
 8   Altitud               9464 non-null   object 
 9   LONGITUD              9464 non-null   float64
 10  LATITUD               9464 non-null   float64
 11  Fecha_instalacion     9331 non-null   object 
 12  Fecha_suspension      3665 non-null   object 
 13  Area Operativa        9464 non-null   object 
 14  Corriente             3156 non-null   object 
 15  Area Hidrografica    

In [ ]:
estaciones_df.head()

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Codigo,Nombre,Categoria,Tecnologia,Estado,Departamento,Municipio,Ubicación,Altitud,LONGITUD,LATITUD,Fecha_instalacion,Fecha_suspension,Area Operativa,Corriente,Area Hidrografica,Zona Hidrografica,Subzona hidrografica,Entidad
0,35060220,LA GLORIA [35060220],Pluviométrica,Convencional,Activa,Cundinamarca,Ubalá,"(-73.41977778, 4.815694444)","1,845",-73.419778,4.815694,15/09/1964,NaN,Area Operativa 11 - Cundinamarca-Amazonas,NaN,Orinoco,Meta,Río Guavio,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...
1,23197510,LADRILLERA [23197510],Limnimétrica,Convencional,Activa,Santander,Bucaramanga,"(-73.13333333, 7.116666667)",880,-73.133333,7.116667,15/09/1981,NaN,Area Operativa 08 - Santanderes-Arauca,QUEBRADA IGLESIA,Magdalena Cauca,Medio Magdalena,Río Lebrija y otros directos al Magdalena,PROYECTO COLOMBO - HOLANDÉS
2,2615700040,AGUAS DE MANIZALES - AUT [2615700040],Limnigráfica,Automática con Telemetría,Activa,Caldas,Manizales,"(-75.490305556, 5.089916667)","1,659",-75.490306,5.089917,04/07/2018,NaN,Area Operativa 09 - Cauca-Valle-Caldas,QUEBRADA EL GUAMO,Magdalena Cauca,Cauca,Río Chinchiná,CORPORACIÓN AUTÓNOMA REGIONAL DE CALDAS
3,26090930,PICHICHI [26090930],Pluviométrica,Convencional,Activa,Valle Del Cauca,Guacarí,"(-76.28333333, 3.783333333)","1,029",-76.283333,3.783333,15/01/1969,NaN,Area Operativa 09 - Cauca-Valle-Caldas,NaN,Magdalena Cauca,Cauca,"Ríos Guabas,Sabaletas y Sonso",ESTACIONES PARTICULARES
4,3503500395,LAGUNA NEGRA AUT - [3503500395],Climatológica Principal,Automática con Telemetría,Activa,Cundinamarca,Fómeque,"(-73.771518889, 4.591235)","3,719",-73.771519,4.591235,11/12/2023,NaN,Area Operativa 11 - Cundinamarca-Amazonas,NaN,Orinoco,Meta,Río Guatiquía,PARQUES NACIONALES NATURALES


In [ ]:
print(estaciones_df['Departamento'].unique())

['Cundinamarca' 'Santander' 'Caldas' 'Valle Del Cauca' 'Bogotá' 'Boyacá'
 'Norte De Santander' 'Atlantico' 'Antioquia' 'Choco' 'Huila' 'Tolima'
 'Quindío' 'Caqueta' 'La Guajira' 'Cauca' 'Risaralda' 'Bolivar'
 'Archipielago De San Andres, Providencia Y Santa Catalina' 'Casanare'
 'Cordoba' 'Amazonas' 'Meta' 'Arauca' 'Magdalena' 'Cesar' 'Putumayo'
 'Sucre' 'Vichada' 'Nariño' 'Guaviare' 'Guainía' 'Vaupes']


In [ ]:
dptos={'Cundinamarca',''}

In [ ]:
estaciones_df['Estado'].value_counts()

,count
Estado,
Activa,5456
Suspendida,3671
En Mantenimiento,337


In [ ]:
departamentos_interes = ['Antioquia', 'Boyacá', 'Cundinamarca']
estaciones_activas_filtradas_df = estaciones_df[
    (estaciones_df['Estado'] == 'Activa') &
    (estaciones_df['Departamento'].isin(departamentos_interes))
]

In [ ]:
print(f"Número de estaciones filtradas: {len(estaciones_activas_filtradas_df)}")

Número de estaciones filtradas: 1924


In [ ]:
estaciones_activas_filtradas_df['Departamento'].value_counts()

,count
Departamento,
Cundinamarca,820
Antioquia,770
Boyacá,334


In [ ]:
estaciones_activas_filtradas_df.to_csv('/content/drive/MyDrive/eco2026/EstacionesActivasIDEAM.csv')

In [ ]:
estaciones_activas=estaciones_activas_filtradas_df['Codigo'].unique()

In [ ]:
datos_estaciones= pd.read_csv('/content/drive/MyDrive/eco2026/Datos_Estaciones_20260527.csv')

/tmp/ipykernel_11367/452111131.py:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  datos_estaciones= pd.read_csv('/content/drive/MyDrive/eco2026/Datos_Estaciones_20260527.csv')


In [ ]:
datos_estaciones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72036 entries, 0 to 72035
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CodigoEstacion     72036 non-null  int64  
 1   CodigoSensor       72036 non-null  object 
 2   FechaObservacion   72036 non-null  object 
 3   ValorObservado     72036 non-null  object 
 4   NombreEstacion     72036 non-null  object 
 5   Departamento       72036 non-null  object 
 6   Municipio          72036 non-null  object 
 7   ZonaHidrografica   72036 non-null  object 
 8   Latitud            72036 non-null  float64
 9   Longitud           72036 non-null  float64
 10  DescripcionSensor  72036 non-null  object 
 11  UnidadMedida       72036 non-null  object 
 12  Entidad            72036 non-null  object 
dtypes: float64(2), int64(1), object(10)
memory usage: 7.1+ MB


In [ ]:
datos_estaciones.head()

,CodigoEstacion,CodigoSensor,FechaObservacion,ValorObservado,NombreEstacion,Departamento,Municipio,ZonaHidrografica,Latitud,Longitud,DescripcionSensor,UnidadMedida,Entidad
0,24035507,0240,2026 May 27 07:00:00 AM,0,SOCHA - AUT [24035507],Boyacá,Socha,Sogamoso,5.982444,-72.710444,Precipitación acumulada 10 minutos,mm,ISAGEN S.A. E.S.P
1,35077090,NVLM_CON,2026 May 27 06:00:00 AM,80,PUENTE ADRIANA [35077090],Boyacá,Jenesano,Meta,5.385000,-73.361639,Nivel del rio a las 600 y 1800 po limnímetro,cm,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...
2,15017020,NVLM_CON,2026 May 27 06:00:00 AM,138,LA REVUELTA [15017020],Magdalena,Santa Marta,Caribe - Guajira,11.277639,-73.941556,Nivel del rio a las 600 y 1800 po limnímetro,cm,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...
3,35097080,NVLM_CON,2026 May 27 06:00:00 AM,324,CEIBAL EL [35097080],Boyacá,Páez,Meta,5.081389,-73.034167,Nivel del rio a las 600 y 1800 po limnímetro,cm,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...
4,15037010,NVLM_CON,2026 May 27 06:00:00 AM,54,ANCHO [15037010],La Guajira,Dibulla,Caribe - Guajira,11.205056,-73.461306,Nivel del rio a las 600 y 1800 po limnímetro,cm,INSTITUTO DE HIDROLOGÍA METEOROLOGÍA Y ESTUDIO...


In [ ]:
datos_estaciones_filtradas_df = datos_estaciones[
    (datos_estaciones['Departamento'].isin(departamentos_interes))
]

In [ ]:
unique_codigo_estacion = datos_estaciones_filtradas_df['CodigoEstacion'].unique()

In [ ]:
print(len(estaciones_activas))
print(len(unique_codigo_estacion))

1924
161


In [ ]:
original_count = len(datos_estaciones_filtradas_df)
datos_estaciones_filtradas_df['FechaObservacion'] = pd.to_datetime(datos_estaciones_filtradas_df['FechaObservacion'], errors='coerce')

converted_count = datos_estaciones_filtradas_df['FechaObservacion'].count()
error_count = original_count - converted_count

print(f"Total de registros: {original_count}")
print(f"Registros transformados exitosamente: {converted_count}")
print(f"Registros con error de transformación: {error_count}")


Total de registros: 22421
Registros transformados exitosamente: 22421
Registros con error de transformación: 0


/tmp/ipykernel_11367/105994947.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  datos_estaciones_filtradas_df['FechaObservacion'] = pd.to_datetime(datos_estaciones_filtradas_df['FechaObservacion'], errors='coerce')


In [ ]:
datos_estaciones_filtradas_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 22421 entries, 0 to 72034
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   CodigoEstacion     22421 non-null  int64         
 1   CodigoSensor       22421 non-null  object        
 2   FechaObservacion   22421 non-null  datetime64[ns]
 3   ValorObservado     22421 non-null  object        
 4   NombreEstacion     22421 non-null  object        
 5   Departamento       22421 non-null  object        
 6   Municipio          22421 non-null  object        
 7   ZonaHidrografica   22421 non-null  object        
 8   Latitud            22421 non-null  float64       
 9   Longitud           22421 non-null  float64       
 10  DescripcionSensor  22421 non-null  object        
 11  UnidadMedida       22421 non-null  object        
 12  Entidad            22421 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(9)
memory usage

In [ ]:
primera_fecha_observacion = datos_estaciones_filtradas_df['FechaObservacion'].min()
ultima_fecha_observacion = datos_estaciones_filtradas_df['FechaObservacion'].max()

print(f"Primera fecha de observación: {primera_fecha_observacion}")
print(f"Última fecha de observación: {ultima_fecha_observacion}")

Primera fecha de observación: 2026-05-27 04:35:00
Última fecha de observación: 2026-05-27 14:00:00
